In [ ]:
# Install dependencies for Kaggle
# Note: Kaggle has torch pre-installed, just need the extras
import subprocess, sys, torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"Python: {sys.version}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

# Core dependencies
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "miditok>=3.0.0",
    "ncps>=0.0.7",
    "safetensors>=0.4.0",
    "pretty_midi>=0.2.10",
    "numpy>=1.24.0",
    "tqdm>=4.65.0",
], check=False)

# Mamba pre-built wheels
# These match PyTorch 2.x + CUDA 12.x which Kaggle uses
# Update wheel URLs if Kaggle updates its PyTorch version
cuda_ver = torch.version.cuda or ""
py_ver = f"{sys.version_info.major}{sys.version_info.minor}"
torch_ver = torch.__version__.split('+')[0]

print(f"\nSelecting Mamba wheel for: CUDA {cuda_ver}, PyTorch {torch_ver}, Python {py_ver}")

# Wheel selection logic - update these URLs when Kaggle updates its stack
WHEEL_MAP = {
    # (cuda_major, torch_major_minor, python_version): (causal_conv1d_url, mamba_url)
    ('12', '2.1', '310'): (
        "https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.4.0/causal_conv1d-1.4.0+cu122torch2.1cxx11abiFALSE-cp310-cp310-linux_x86_64.whl",
        "https://github.com/state-spaces/mamba/releases/download/v2.2.2/mamba_ssm-2.2.2+cu122torch2.1cxx11abiFALSE-cp310-cp310-linux_x86_64.whl"
    ),
    ('12', '2.1', '311'): (
        "https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.4.0/causal_conv1d-1.4.0+cu122torch2.1cxx11abiFALSE-cp311-cp311-linux_x86_64.whl",
        "https://github.com/state-spaces/mamba/releases/download/v2.2.2/mamba_ssm-2.2.2+cu122torch2.1cxx11abiFALSE-cp311-cp311-linux_x86_64.whl"
    ),
}

# Try to find a matching wheel
cuda_major = cuda_ver.split('.')[0] if cuda_ver else ''
torch_major_minor = '.'.join(torch_ver.split('.')[:2])

wheel_key = (cuda_major, torch_major_minor, py_ver)
if wheel_key in WHEEL_MAP:
    causal_url, mamba_url = WHEEL_MAP[wheel_key]
    r1 = subprocess.run([sys.executable, "-m", "pip", "install", "-q", causal_url], capture_output=True)
    r2 = subprocess.run([sys.executable, "-m", "pip", "install", "-q", mamba_url], capture_output=True)
    if r2.returncode == 0:
        print("Mamba installed from pre-built wheel")
    else:
        print(f"Wheel install failed: {r2.stderr.decode()[-300:]}")
        print("Falling back to PyPI (may compile from source)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mamba-ssm"], check=False)
else:
    print(f"No pre-built wheel for {wheel_key}")
    print("Run this cell first to get exact versions:")
    print("  import torch, sys")
    print("  print(torch.version.cuda, torch.__version__, sys.version)")
    print("Then find the matching wheel at:")
    print("  https://github.com/state-spaces/mamba/releases")
    print("  https://github.com/Dao-AILab/causal-conv1d/releases")
    print("Falling back to GRU for now...")

# Verify
try:
    import mamba_ssm
    print(f"mamba-ssm {mamba_ssm.__version__} active")
except ImportError:
    print("mamba-ssm not available - GRU fallback will be used")


In [ ]:
# ============================================
# EDIT THIS CELL TO CHANGE TRAINING SETTINGS
# ============================================
SCALE = "nano"        # Options: nano, micro, small, medium
MAX_EPOCHS = 2000     # Total target epochs across all sessions
# ============================================

# Show available presets
import sys
sys.path.insert(0, '/kaggle/input/piano-midi-model')  # update slug if needed
from scale_config import list_presets, get_preset
list_presets()
print(f"\nSelected: {SCALE}")
get_preset(SCALE)


In [ ]:
from kaggle_session import run_kaggle_session
run_kaggle_session(scale=SCALE, max_epochs=MAX_EPOCHS)


In [ ]:
# Run this to check actual parameter counts on this GPU runtime
from kaggle_session import calibrate_on_kaggle
calibrate_on_kaggle()


In [ ]:
# Generate a continuation from the best checkpoint
from kaggle_session import generate_from_best_checkpoint
generate_from_best_checkpoint(scale=SCALE)


## Downloading your trained model

After training, download your checkpoints from the Kaggle output panel:
1. Click the three dots (...) next to the notebook
2. Select "Output"
3. Download `checkpoints/best.safetensors` - this is your best model
4. Download `checkpoints/latest.safetensors` - this is the most recent epoch

To continue training in a future session:
- Re-run this notebook
- It will automatically detect and resume from the latest checkpoint
- Checkpoints persist as long as you save the notebook version

To use your model locally:
```bash
python tools/generate_sample.py \
  --checkpoint path/to/best.safetensors \
  --seed your_seed.mid \
  --output generated.mid \
  --scale nano
```
